# 04 · Report Figures

This notebook generates clean figures and summary tables for the final report and presentation of the Komoot trail recommender project.

It is intentionally separated from the main project notebooks so that the report/presentation work can continue in parallel without modifying:

- `01_data_exploration.ipynb`
- `02_task1_nlp.ipynb`
- `03_task2_recommender.ipynb`

The notebook reads the processed dataset `trails.pkl` and exports figures to `report/figures/` and tables to `report/tables/`.

ModuleNotFoundError: No module named 'pandas'

## 1. Path configuration

The code below tries several possible locations for `trails.pkl`, so it can be executed either from the repository root, from the `notebooks/` folder, or from this sandbox.

In [ ]:
# Candidate locations depending on where the notebook is executed from.
DATA_CANDIDATES = [
    Path("data/trails.pkl"),
    Path("../data/trails.pkl"),
    Path("trails.pkl"),
    Path("/mnt/data/trails.pkl"),
]

DATA_PATH = None
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate.resolve()
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find trails.pkl. Put it in data/trails.pkl, ../data/trails.pkl, "
        "or update DATA_CANDIDATES with the correct path."
    )

# Infer a reasonable project root.
cwd = Path.cwd().resolve()
if (cwd / "data" / "trails.pkl").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "trails.pkl").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

FIG_DIR = PROJECT_ROOT / "report" / "figures"
TABLE_DIR = PROJECT_ROOT / "report" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using dataset: {DATA_PATH}")
print(f"Figures will be saved to: {FIG_DIR}")
print(f"Tables will be saved to: {TABLE_DIR}")

## 2. Load dataset

In [ ]:
df = pd.read_pickle(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} trails × {df.shape[1]:,} columns")
display(df.head(3))

## 3. Create derived variables for plotting

Distances, durations and elevations are converted into more readable units for the report.

In [ ]:
df_plot = df.copy()

numeric_columns = [
    "distance_m",
    "duration_s",
    "elevation_up_m",
    "elevation_down_m",
    "rating_score",
    "rating_count",
    "visitors",
]

for col in numeric_columns:
    if col in df_plot.columns:
        df_plot[col] = pd.to_numeric(df_plot[col], errors="coerce")

if "distance_m" in df_plot.columns:
    df_plot["distance_km"] = df_plot["distance_m"] / 1000

if "duration_s" in df_plot.columns:
    df_plot["duration_h"] = df_plot["duration_s"] / 3600

if {"distance_km", "duration_h"}.issubset(df_plot.columns):
    df_plot["avg_speed_estimated_kmh"] = df_plot["distance_km"] / df_plot["duration_h"].replace(0, np.nan)

print("Available columns:")
print(df_plot.columns.tolist())

## 4. Dataset summary tables

These tables can be exported directly to the report or used to ensure that the written report uses consistent numbers.

In [ ]:
summary_rows = []

summary_rows.append(("Number of trails", len(df_plot)))
summary_rows.append(("Number of columns", df_plot.shape[1]))

if "tour_id" in df_plot.columns:
    summary_rows.append(("Unique tour IDs", df_plot["tour_id"].nunique()))

if "description" in df_plot.columns:
    summary_rows.append(("Trails with non-empty description", df_plot["description"].notna().sum()))

if "sport" in df_plot.columns:
    summary_rows.append(("Number of sports", df_plot["sport"].nunique(dropna=True)))

if "region" in df_plot.columns:
    summary_rows.append(("Number of regions", df_plot["region"].nunique(dropna=True)))

if "rating_score" in df_plot.columns:
    summary_rows.append(("Trails with rating_score", df_plot["rating_score"].notna().sum()))
    summary_rows.append(("Trails without rating_score", df_plot["rating_score"].isna().sum()))

dataset_summary = pd.DataFrame(summary_rows, columns=["Metric", "Value"])
display(dataset_summary)

dataset_summary.to_csv(TABLE_DIR / "dataset_summary.csv", index=False)
print(f"Saved: {TABLE_DIR / 'dataset_summary.csv'}")

In [ ]:
important_columns = [
    "tour_id",
    "name",
    "sport",
    "difficulty",
    "distance_m",
    "duration_s",
    "elevation_up_m",
    "elevation_down_m",
    "region",
    "description",
    "rating_score",
    "rating_count",
    "visitors",
    "categories",
    "surfaces",
    "way_types",
    "waypoints",
]

existing_important_columns = [col for col in important_columns if col in df_plot.columns]

missing_table = (
    df_plot[existing_important_columns]
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "Column", 0: "Missing values"})
)

missing_table["Missing percentage"] = (
    missing_table["Missing values"] / len(df_plot) * 100
).round(2)

missing_table = missing_table.sort_values("Missing values", ascending=False)

display(missing_table)

missing_table.to_csv(TABLE_DIR / "missing_values_important_columns.csv", index=False)
print(f"Saved: {TABLE_DIR / 'missing_values_important_columns.csv'}")

In [ ]:
numeric_report_columns = [
    "distance_km",
    "duration_h",
    "elevation_up_m",
    "elevation_down_m",
    "rating_score",
    "rating_count",
    "visitors",
]

existing_numeric_report_columns = [col for col in numeric_report_columns if col in df_plot.columns]

numeric_summary = (
    df_plot[existing_numeric_report_columns]
    .describe()
    .T
    .round(3)
)

display(numeric_summary)

numeric_summary.to_csv(TABLE_DIR / "numeric_summary.csv")
print(f"Saved: {TABLE_DIR / 'numeric_summary.csv'}")

## 5. Plot helper functions

In [ ]:
def save_fig(filename: str):
    """Save the current matplotlib figure with a consistent configuration."""
    path = FIG_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")
    return path


def plot_value_counts(series, title, xlabel, ylabel, filename, top_n=None, rotation=45):
    """Plot a bar chart from a categorical series."""
    data = series.fillna("Missing").astype(str).value_counts()
    if top_n is not None:
        data = data.head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    data.plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=rotation)
    save_fig(filename)
    return data


def plot_hist(series, title, xlabel, ylabel, filename, bins=30, clip_quantile=None):
    """Plot a histogram for a numeric variable, optionally clipping extreme outliers."""
    data = pd.to_numeric(series, errors="coerce").dropna()

    if clip_quantile is not None and not data.empty:
        upper = data.quantile(clip_quantile)
        data = data[data <= upper]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(data, bins=bins)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    save_fig(filename)
    return data

## 6. Categorical distributions

In [ ]:
if "sport" in df_plot.columns:
    sport_counts = plot_value_counts(
        df_plot["sport"],
        title="Distribution of trails by sport",
        xlabel="Sport",
        ylabel="Number of trails",
        filename="sport_distribution.png",
        rotation=45,
    )

    display(sport_counts.to_frame("Number of trails"))
    sport_counts.to_csv(TABLE_DIR / "sport_distribution.csv", header=["Number of trails"])

In [ ]:
if "difficulty" in df_plot.columns:
    difficulty_counts = plot_value_counts(
        df_plot["difficulty"],
        title="Distribution of trails by difficulty",
        xlabel="Difficulty",
        ylabel="Number of trails",
        filename="difficulty_distribution.png",
        rotation=0,
    )

    display(difficulty_counts.to_frame("Number of trails"))
    difficulty_counts.to_csv(TABLE_DIR / "difficulty_distribution.csv", header=["Number of trails"])

In [ ]:
if "tour_type" in df_plot.columns:
    tour_type_counts = plot_value_counts(
        df_plot["tour_type"],
        title="Distribution of trails by tour type",
        xlabel="Tour type",
        ylabel="Number of trails",
        filename="tour_type_distribution.png",
        rotation=45,
    )

    display(tour_type_counts.to_frame("Number of trails"))
    tour_type_counts.to_csv(TABLE_DIR / "tour_type_distribution.csv", header=["Number of trails"])

In [ ]:
if "roundtrip" in df_plot.columns:
    roundtrip_counts = plot_value_counts(
        df_plot["roundtrip"],
        title="Roundtrip vs non-roundtrip trails",
        xlabel="Roundtrip",
        ylabel="Number of trails",
        filename="roundtrip_distribution.png",
        rotation=0,
    )

    display(roundtrip_counts.to_frame("Number of trails"))
    roundtrip_counts.to_csv(TABLE_DIR / "roundtrip_distribution.csv", header=["Number of trails"])

## 7. Numerical distributions

For distance, duration, elevation, rating count and visitors, the plots use the 99th percentile as a clipping threshold. This avoids a few extreme routes dominating the chart.

In [ ]:
if "distance_km" in df_plot.columns:
    distance_data = plot_hist(
        df_plot["distance_km"],
        title="Distribution of trail distance",
        xlabel="Distance (km)",
        ylabel="Number of trails",
        filename="distance_distribution_km.png",
        bins=40,
        clip_quantile=0.99,
    )
    print(distance_data.describe().round(2))

In [ ]:
if "duration_h" in df_plot.columns:
    duration_data = plot_hist(
        df_plot["duration_h"],
        title="Distribution of estimated trail duration",
        xlabel="Duration (hours)",
        ylabel="Number of trails",
        filename="duration_distribution_hours.png",
        bins=40,
        clip_quantile=0.99,
    )
    print(duration_data.describe().round(2))

In [ ]:
if "elevation_up_m" in df_plot.columns:
    elevation_data = plot_hist(
        df_plot["elevation_up_m"],
        title="Distribution of positive elevation gain",
        xlabel="Elevation gain (m)",
        ylabel="Number of trails",
        filename="elevation_up_distribution_m.png",
        bins=40,
        clip_quantile=0.99,
    )
    print(elevation_data.describe().round(2))

In [ ]:
if "rating_score" in df_plot.columns:
    rating_data = plot_hist(
        df_plot["rating_score"],
        title="Distribution of available rating scores",
        xlabel="Rating score",
        ylabel="Number of trails",
        filename="rating_score_distribution.png",
        bins=20,
        clip_quantile=None,
    )
    print(rating_data.describe().round(2))
    print(f"Missing rating_score values: {df_plot['rating_score'].isna().sum():,}")

In [ ]:
if "rating_count" in df_plot.columns:
    rating_count_log = np.log1p(pd.to_numeric(df_plot["rating_count"], errors="coerce"))
    rating_count_data = plot_hist(
        rating_count_log,
        title="Distribution of rating counts (log scale)",
        xlabel="log(1 + rating_count)",
        ylabel="Number of trails",
        filename="rating_count_log_distribution.png",
        bins=40,
        clip_quantile=0.99,
    )
    print(rating_count_data.describe().round(2))

In [ ]:
if "visitors" in df_plot.columns:
    visitors_log = np.log1p(pd.to_numeric(df_plot["visitors"], errors="coerce"))
    visitors_data = plot_hist(
        visitors_log,
        title="Distribution of visitors (log scale)",
        xlabel="log(1 + visitors)",
        ylabel="Number of trails",
        filename="visitors_log_distribution.png",
        bins=40,
        clip_quantile=0.99,
    )
    print(visitors_data.describe().round(2))

## 8. Regional distribution

In [ ]:
if "region" in df_plot.columns:
    region_counts = plot_value_counts(
        df_plot["region"],
        title="Top 20 regions by number of trails",
        xlabel="Region",
        ylabel="Number of trails",
        filename="top_regions_distribution.png",
        top_n=20,
        rotation=75,
    )

    display(region_counts.to_frame("Number of trails"))
    region_counts.to_csv(TABLE_DIR / "top_regions_distribution.csv", header=["Number of trails"])

## 9. Cross-tabulations for the report

These tables are useful for explaining the composition of the dataset.

In [ ]:
if {"sport", "difficulty"}.issubset(df_plot.columns):
    sport_difficulty = pd.crosstab(df_plot["sport"], df_plot["difficulty"])
    display(sport_difficulty)

    sport_difficulty.to_csv(TABLE_DIR / "sport_by_difficulty.csv")
    print(f"Saved: {TABLE_DIR / 'sport_by_difficulty.csv'}")

    fig, ax = plt.subplots(figsize=(10, 6))
    sport_difficulty.plot(kind="bar", stacked=True, ax=ax)
    ax.set_title("Sport distribution by difficulty")
    ax.set_xlabel("Sport")
    ax.set_ylabel("Number of trails")
    ax.tick_params(axis="x", rotation=45)
    save_fig("sport_by_difficulty_stacked.png")

## 10. Categories

The `categories` field contains lists, so it needs to be flattened before plotting.

In [ ]:
def ensure_list(value):
    """Convert a value into a list when possible."""
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            return []
    return []


if "categories" in df_plot.columns:
    category_counter = Counter()

    for value in df_plot["categories"].dropna():
        category_counter.update(ensure_list(value))

    category_counts = pd.Series(category_counter).sort_values(ascending=False)

    if not category_counts.empty:
        top_category_counts = category_counts.head(25)

        fig, ax = plt.subplots(figsize=(10, 7))
        top_category_counts.plot(kind="bar", ax=ax)
        ax.set_title("Top 25 trail categories")
        ax.set_xlabel("Category")
        ax.set_ylabel("Frequency")
        ax.tick_params(axis="x", rotation=75)
        save_fig("top_categories_distribution.png")

        display(top_category_counts.to_frame("Frequency"))
        top_category_counts.to_csv(TABLE_DIR / "top_categories_distribution.csv", header=["Frequency"])
    else:
        print("No categories found.")

## 11. Surface and way type composition

The `surfaces` and `way_types` fields are dictionaries containing proportions. The charts below show the average proportion of each type across the dataset.

In [ ]:
def parse_dict_field(value):
    """Convert dictionary-like values into a Python dict."""
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            return {}
    return {}


def average_dict_proportions(series):
    accumulator = Counter()
    n_valid = 0

    for value in series.dropna():
        parsed = parse_dict_field(value)
        if parsed:
            for key, val in parsed.items():
                try:
                    accumulator[key] += float(val)
                except Exception:
                    pass
            n_valid += 1

    if n_valid == 0:
        return pd.Series(dtype=float)

    return (pd.Series(accumulator) / n_valid).sort_values(ascending=False)


if "surfaces" in df_plot.columns:
    avg_surfaces = average_dict_proportions(df_plot["surfaces"])

    if not avg_surfaces.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        avg_surfaces.plot(kind="bar", ax=ax)
        ax.set_title("Average surface composition")
        ax.set_xlabel("Surface type")
        ax.set_ylabel("Average proportion")
        ax.tick_params(axis="x", rotation=45)
        save_fig("average_surface_composition.png")

        display(avg_surfaces.to_frame("Average proportion"))
        avg_surfaces.to_csv(TABLE_DIR / "average_surface_composition.csv", header=["Average proportion"])
    else:
        print("No surface data found.")

In [ ]:
if "way_types" in df_plot.columns:
    avg_way_types = average_dict_proportions(df_plot["way_types"])

    if not avg_way_types.empty:
        fig, ax = plt.subplots(figsize=(10, 6))
        avg_way_types.plot(kind="bar", ax=ax)
        ax.set_title("Average way type composition")
        ax.set_xlabel("Way type")
        ax.set_ylabel("Average proportion")
        ax.tick_params(axis="x", rotation=45)
        save_fig("average_way_type_composition.png")

        display(avg_way_types.to_frame("Average proportion"))
        avg_way_types.to_csv(TABLE_DIR / "average_way_type_composition.csv", header=["Average proportion"])
    else:
        print("No way type data found.")

## 12. Candidate figures for the final report

Recommended figures to include in the report:

1. `sport_distribution.png`
2. `difficulty_distribution.png`
3. `distance_distribution_km.png`
4. `rating_score_distribution.png`
5. `top_regions_distribution.png`
6. `sport_by_difficulty_stacked.png`
7. `top_categories_distribution.png`
8. `average_surface_composition.png`

Recommended tables:

1. `dataset_summary.csv`
2. `missing_values_important_columns.csv`
3. `numeric_summary.csv`
4. `sport_distribution.csv`
5. `sport_by_difficulty.csv`